# Mooncake Store 在昇腾 NPU 上的使用

本节将介绍 Mooncake Store 在昇腾 NPU 上的 Python API 使用以及完整的使用流程，帮助你掌握跨节点 KV Cache 共享的实践方法。

本节学习大纲如下：

- API 参考与注意事项
- 完整示例（三种部署模式）


## 1. API 参考与注意事项

### 1.1 核心方法

| 方法 | 说明 | 何时使用 |
| --- | --- | --- |
| `setup(...)` | 初始化 Store | Embedded / Standalone |
| `setup_dummy(...)` | 初始化 Dummy 客户端 | Dummy/Real 模式 |
| `register_buffer(addr, size)` | 注册 NPU/Host 缓冲区 | put/get 前必须调用 |
| `batch_put_from(keys, ptrs, sizes)` | 批量写入（D2H） | Prefill 写入 KV Cache |
| `batch_get_into(keys, ptrs, sizes)` | 批量读取（H2D） | Decode 读取 KV Cache |
| `close()` | 关闭 Store | 程序退出前 |

### 1.2 setup 参数说明

| 参数 | 说明 | Embedded | Standalone | Dummy |
| --- | --- | --- | --- | --- |
| `local_hostname` | 本机 IP | ✓ | ✓ | 不需要 |
| `metadata_server` | Master 的 HTTP 地址 | ✓ | ✓ | 不需要 |
| `global_segment_size` | 贡献到全局池的内存大小 | > 0 | 0 | 不需要 |
| `local_buffer_size` | 本地缓冲区大小 | ✓ | ✓ | 不需要 |
| `protocol` | 传输协议，昇腾用 `"ascend"` | ✓ | ✓ | 不需要 |
| `rdma_devices` | 网卡设备名，空=自动发现 | ✓ | ✓ | 不需要 |
| `master_server_addr` | Master 的 RPC 地址 | ✓ | ✓ | 不需要 |
| `server_address` | Real Client 的 RPC 地址 | 不需要 | 不需要 | ✓ |

> 通过框架部署时，以上参数通过 `mooncake_config.json` 配置文件传入，由 connector 自动调用，无需手写。详见 Section 2。

### 1.3 使用注意事项

**register_buffer 必须在 put/get 前调用**：告诉 Transfer Engine 哪些内存可以读写。不注册会报错。HCCS 传输要求首地址 2MB 对齐，建议用大页内存（`MC_STORE_USE_HUGEPAGE=1`）。

**Key 命名**：Store 是 key-value 存储，key 自定义。多层 KV Cache 建议 `模型名_batch号_层号` 格式，如 `llama7b_batch0_layer0`。

**同节点传输优化**：当目标 segment 在同一台机器上时，Transfer Engine 自动使用 `aclrtMemcpy` 直接拷贝（比走 HIXL/RDMA 更快），不经过 HIXL。跨节点传输才走 HIXL（HCCS/RoCE/FabricMem）。

### 1.4 传输链路与 HIXL 集成

Mooncake 通过 `protocol="ascend"` 自动调用 HIXL 能力。底层物理链路取决于硬件和环境变量：

| 链路 | 介质 | 带宽 | 适用场景 |
| --- | --- | --- | --- |
| HCCS | 超节点统一编址 | ~100 GB/s | A2/A3 机内；支持 D2D/H2D/D2H |
| RoCE (RDMA) | RoCE 网卡 | ~20 GB/s | A2/A3 跨机；支持 Host DRAM 池化 |
| FabricMem | 灵渠/HCCS 统一编址 + VMM | ~100 GB/s | A3 专用；支持 Host/Device 远端读写 |

> **Store 池化场景**：HCCS（机内）、RoCE（跨机）、FabricMem（A3）均可用于 Store 池化。HCCS 是 A2/A3 机内的默认协议。

**链路选择策略**：A2/A3 机内默认 HCCS；A3 推荐 FabricMem（HDK≥25.5, CANN≥9.0）；跨机用 RoCE。设 `HCCL_INTRA_ROCE_ENABLE=1` 可强制机内也走 RoCE。

**关键环境变量**：

| 环境变量 | 说明 |
| --- | --- |
| `ASCEND_ENABLE_USE_FABRIC_MEM=1` | A3 启用 FabricMem |
| `HCCL_INTRA_ROCE_ENABLE=1` | 强制使用 RoCE |
| `MC_STORE_USE_HUGEPAGE=1` | 使用大页内存 |
| `ASCEND_CONNECT_TIMEOUT` | 连接超时（秒），默认 300 |
| `ASCEND_TRANSFER_TIMEOUT` | 传输超时（秒），默认 60 |



## 2. 完整示例

本节展示三种部署模式的使用方式。**Embedded 模式提供完整可运行脚本**，Dummy/Real 和 Standalone 模式仅介绍部署方式和关键区别（学生只需了解部署方式和区别）。

> **前置条件**：所有模式都需要先启动 Master。Embedded 模式的 Master 启动已封装在下方可执行单元格中（步骤 2）；Dummy/Real 和 Standalone 模式手动启动 Master 的命令如下：
```bash
mooncake_master \
    --port=50051 \
    --enable_http_metadata_server=true \
    --http_metadata_server_host=0.0.0.0 \
    --http_metadata_server_port=8080
```

### 2.1 Embedded 模式（可运行）

Store 内嵌在推理脚本里，推理进程自己贡献内存、自己传数据。只需启动 Master + 运行脚本，无需额外进程，适合快速验证。

<img src="./images/embedded_deploy.svg" alt="Embedded 模式部署" width="90%">

完整流程已拆解为下方 4 个可执行单元格（步骤 1~4）：设置环境 → 启动 Master → 运行写入/读取 → 停止 Master。等价的完整脚本如下，供参考：

```python
# embedded_demo.py（等价完整脚本）
from mooncake.store import MooncakeDistributedStore
import torch
torch.npu.set_device(0)  # 设置 NPU 设备

store = MooncakeDistributedStore()
store.setup(
    local_hostname="127.0.0.1",
    metadata_server="http://127.0.0.1:8080/metadata",
    global_segment_size=1024 * 1024 * 1024,  # 贡献 1GB
    local_buffer_size=0,
    protocol="ascend",
    rdma_devices="",
    master_server_addr="127.0.0.1:50051"
)

# 创建 NPU tensor 并注册
tensor = torch.ones(1024 * 1024, dtype=torch.float16, device="npu")  # 2MB
store.register_buffer(tensor.data_ptr(), tensor.nbytes)

# 写入 Store（D2H）
results = store.batch_put_from(["kv_cache_0"], [tensor.data_ptr()], [tensor.nbytes])
print(f"Put: {'OK' if results[0] == 0 else 'FAIL'}")

# 读取 Store（H2D）
dst = torch.zeros(1024 * 1024, dtype=torch.float16, device="npu")
store.register_buffer(dst.data_ptr(), dst.nbytes)
results = store.batch_get_into(["kv_cache_0"], [dst.data_ptr()], [dst.nbytes])
print(f"Get result: {results[0]}")  # 正数=成功（字节数），负数=错误码

# 验证
if results[0] > 0:
    print(f"Data matches: {torch.equal(tensor, dst)}")
else:
    print(f"Get failed, error code: {results[0]}")

store.close()
```

预期输出：
```
Put: OK
Get result: 2097152
Data matches: True
```

### 2.2 Dummy/Real 模式（仅介绍）

一机多卡场景（TP>1）专用。一个 Real Client 进程独占管理所有 NPU 卡的内存池和传输，多个推理进程作为 Dummy Client 通过 RPC 调用它，自己不创建 TE。适合多卡共享一个 HIXL 实例、减少资源开销。

<img src="./images/dummy_real_deploy.svg" alt="Dummy/Real 部署流程" width="90%">

**部署方式**：
1. 启动 Real Client（管理所有卡，贡献内存池）：
   ```bash
   export ASCEND_RT_VISIBLE_DEVICES=0,1
   mooncake_client \
       --host=127.0.0.1 \
       --global_segment_size=4GB \
       --master_server_address=127.0.0.1:50051 \
       --metadata_server=http://127.0.0.1:8080/metadata \
       --protocol=ascend
   ```
2. 推理进程用 `setup_dummy` 替代 `setup`，后续 register/put/get/close 完全相同：
   ```python
   store.setup_dummy(
       mem_pool_size=0,
       local_buffer_size=0,
       server_address="127.0.0.1:50052"  # Real Client 的 RPC 端口
   )
   ```

**关键区别**：Dummy 进程没有 TE/HIXL，通过 coro_rpc 转发给 Real Client 执行。Real Client 收到请求后通过自己的 HIXL 实例执行传输，直接读写 Dummy 进程共享的 NPU 显存。

### 2.3 Standalone 模式（仅介绍）

内存与推理解耦。`mooncake_client` 独立进程贡献内存池，推理进程以 `global_segment_size=0` 纯客户端接入，但保留自己的 TE 做数据传输。推理进程退出后缓存不丢失，适合生产部署。

<img src="./images/standalone_deploy.svg" alt="Standalone 模式部署" width="90%">

**部署方式**：
1. 启动 mooncake_client（贡献内存，需用不同卡）：
   ```bash
   export ASCEND_RT_VISIBLE_DEVICES=1
   mooncake_client \
       --host=127.0.0.1 \
       --global_segment_size=4GB \
       --master_server_address=127.0.0.1:50051 \
       --metadata_server=http://127.0.0.1:8080/metadata \
       --protocol=ascend
   ```
2. 推理进程用 `setup` 但 `global_segment_size=0`：
   ```python
   store.setup(
       local_hostname="127.0.0.1",
       metadata_server="http://127.0.0.1:8080/metadata",
       global_segment_size=0,  # 纯客户端，不贡献内存
       local_buffer_size=0,
       protocol="ascend",
       rdma_devices="",
       master_server_addr="127.0.0.1:50051"
   )
   ```

**关键区别**：推理进程有自己的 TE + HIXL，数据通过 HIXL 传到 mooncake_client 的内存。

> **三种模式对比**：只有初始化方式不同，后续 register/put/get/close 完全相同。

| | Embedded | Standalone | Dummy/Real |
| --- | --- | --- | --- |
| TE 实例 | 推理进程内 | 推理进程内 | Real Client 独立进程 |
| 内存贡献 | 推理进程贡献 | mooncake_client 贡献 | Real Client 贡献 |
| 进程退出后数据 | 丢失 | 保留 | 保留 |
| 适用场景 | 快速验证 | 生产部署 | 一机多卡(TP>1) |


## 3. 动手实践：Embedded 模式全流程

以下 4 个单元格按顺序执行，在 Jupyter 中完成 Master 启动 → KV 写入/读取 → Master 停止的完整流程。

> 注意：每个 `!` 命令行都是独立子 shell，环境变量请用 `os.environ` 设置（对后续所有单元格生效），不要用 `!export` 单独一行。

In [ ]:
# 步骤 1：环境准备
import os

# mooncake_master 入口安装在 ~/.local/bin（pip --user 安装），追加到 PATH
os.environ["PATH"] = f"{os.environ['PATH']}:{os.path.expanduser('~/.local/bin')}"

# 规避 RoCE endpoint 自动生成（无 hccn.conf 的容器环境）
os.environ["ASCEND_LOCAL_COMM_RES"] = '{"version":"1.2"}'

print(os.environ["PATH"])

In [ ]:
# 步骤 2：后台启动 mooncake_master（等待就绪，日志落盘可查）
import subprocess, time, socket

master_proc = subprocess.Popen(
    ["mooncake_master",
     "--port=50051",
     "--enable_http_metadata_server=true",
     "--http_metadata_server_host=0.0.0.0",
     "--http_metadata_server_port=8080"],
    stdout=open("/tmp/mooncake_master.log", "w"),
    stderr=subprocess.STDOUT,
)

def port_open(host, port):
    try:
        with socket.create_connection((host, port), timeout=2):
            return True
    except OSError:
        return False

# 轮询 gRPC(50051) 与 HTTP 元数据(8080) 端口直到可连通
for i in range(30):
    time.sleep(1)
    if port_open("127.0.0.1", 8080) and port_open("127.0.0.1", 50051):
        print(f"master 就绪（{i + 1}s）")
        break
else:
    print("master 30s 未就绪，最近日志：")
    print(open("/tmp/mooncake_master.log").read()[-1500:])

In [ ]:
%%writefile store_demo.py
# 步骤 3：写入 Store（D2H）并读取验证（H2D）
from mooncake.store import MooncakeDistributedStore
import torch
import torch_npu

torch.npu.set_device(0)  # 逻辑设备号，容器内为 0（不是物理卡号）

store = MooncakeDistributedStore()
ret = store.setup(
    local_hostname="127.0.0.1",
    metadata_server="http://127.0.0.1:8080/metadata",
    global_segment_size=1024 * 1024 * 1024,  # 贡献 1GB
    local_buffer_size=0,
    protocol="ascend",
    rdma_devices="",
    master_server_addr="127.0.0.1:50051",
)
print(f"setup ret: {ret}")  # 0 表示成功

# 创建 NPU tensor 并注册
tensor = torch.ones(1024 * 1024, dtype=torch.float16, device="npu")  # 2MB
store.register_buffer(tensor.data_ptr(), tensor.nbytes)

# 写入 Store（D2H）
results = store.batch_put_from(["kv_cache_0"], [tensor.data_ptr()], [tensor.nbytes])
print(f"Put: {'OK' if results[0] == 0 else 'FAIL'}")

# 读取 Store（H2D）
dst = torch.zeros(1024 * 1024, dtype=torch.float16, device="npu")
store.register_buffer(dst.data_ptr(), dst.nbytes)
results = store.batch_get_into(["kv_cache_0"], [dst.data_ptr()], [dst.nbytes])
print(f"Get result: {results[0]}")  # 正数=成功（字节数），负数=错误码

# 验证
if results[0] > 0:
    print(f"Data matches: {torch.equal(tensor, dst)}")
else:
    print(f"Get failed, error code: {results[0]}")

store.close()

In [ ]:
# 执行 demo（! 子进程会继承步骤 1 在内核中设置的 PATH 与环境变量）
!python3 store_demo.py

In [ ]:
# 步骤 4：停止 master
master_proc.terminate()
print("master 已停止:", master_proc.poll() is not None)

## 课后练习

1. （判断题）使用 Mooncake Store 进行零拷贝读写前，必须先调用 `register_buffer` 注册缓冲区。

2. （判断题）Mooncake Store 的 `batch_put_from` 返回后，所有 key 的传输一定都成功了。

3. （单选题）Mooncake Store 的 `setup` 方法中，`protocol` 参数在昇腾 NPU 场景下应设为？
    A. `"tcp"`
    B. `"ascend"`
    C. `"rdma"`
    D. `"hccs"`

4. （单选题）A3 场景下推荐的池化方案是？
    A. RoCE 池化
    B. FabricMem 池化
    C. HCCS 池化
    D. UB 池化

5. （单选题）Dummy/Real 模式中，Dummy Client 的 `mem_pool_size` 参数应设为？
    A. 与 Real Client 相同
    B. 0
    C. -1
    D. 任意值都行

6. （单选题）Standalone 和 Dummy/Real 模式的关键区别是？
    A. Standalone 推理进程没有 TE，Dummy/Real 有
    B. Standalone 推理进程有自己的 TE，Dummy 的 Dummy 没有 TE
    C. 两者都没有 TE
    D. 两者 TE 都在 Real Client

7. （多选题）关于 HCCS 传输链路，以下说法正确的是？
    A. 仅支持跨机通信
    B. 适用于 PD 分离的 D2D 直传
    C. 仅支持 Device 内存传输，不支持 Host 内存
    D. 是 A2/A3 机内的芯片间互联

8. （多选题）使用 Mooncake Store 时，以下哪些说法正确？
    A. HCCS 传输要求注册内存首地址 2MB 对齐
    B. 建议生产环境使用 Standalone 或 Dummy/Real 模式
    C. `batch_put_from` 返回值需要检查，部分 key 可能失败
    D. Dummy 模式需要先启动 Real Client

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/06.04_answer.txt